**Import + load Phase 4 JSON**

In [9]:
import json

PHASE4_JSON = "/kaggle/input/datasets/rkr2001/phase-4-output2/phase4_nutrition.json"  

with open(PHASE4_JSON, "r") as f:
    phase4 = json.load(f)

totals = phase4["totals"]
items  = phase4["items"]

print("Loaded totals:", totals)
print("No. of items:", len(items))


Loaded totals: {'kcal': 440.53, 'protein_g': 29.34, 'fat_g': 2.48, 'carbs_g': 78.79, 'sugar_g': 18.45, 'fiber_g': 15.11}
No. of items: 4


**Helper function to safely read totals**

In [10]:
def get_float(d, key, default=0.0):
    v = d.get(key, default)
    return float(v) if v is not None else float(default)

kcal    = get_float(totals, "kcal")
carbs   = get_float(totals, "carbs_g")
fiber   = get_float(totals, "fiber_g")
sugar   = get_float(totals, "sugar_g")     # usually TOTAL sugar
protein = get_float(totals, "protein_g")
fat     = get_float(totals, "fat_g")

net_carbs = round(max(carbs - fiber, 0), 2)

print("kcal:", kcal, "carbs:", carbs, "fiber:", fiber, "net_carbs:", net_carbs)


kcal: 440.53 carbs: 78.79 fiber: 15.11 net_carbs: 63.68


**Define rule thresholds (tune later)**

In [11]:
RULES = {
    # Blood sugar (net carbs is the most useful signal)
    "net_carbs_high": 60,        # warning
    "net_carbs_very_high": 80,   # strong warning
    "sugar_high": 25,            # warning (note: total sugar)
    "fiber_good": 8,             # positive

    # Cholesterol (your dataset lacks sat-fat & dietary cholesterol mg, so use proxies)
    "fiber_chol_good": 10,       # positive
    "fat_high": 25,              # warning proxy
    "protein_low": 20            # tip
}


**Create the rule engine (this generates messages)**

In [12]:
def apply_rules_for_bloodsugar_cholesterol(
    kcal, carbs, fiber, sugar, protein, fat,
    diabetes=False, cholesterol=False
):
    recs = []
    scores = {"blood_sugar": 0, "cholesterol": 0}

    net_carbs = max(carbs - fiber, 0)

    # Always show base summary
    recs.append({
        "type": "info",
        "message": f"Meal summary: {kcal:.0f} kcal | Net carbs: {net_carbs:.1f} g | Protein: {protein:.1f} g | Fiber: {fiber:.1f} g | Fat: {fat:.1f} g"
    })

    # -------- Blood sugar rules --------
    if diabetes:
        if net_carbs >= RULES["net_carbs_very_high"]:
            recs.append({
                "type": "warning",
                "message": "For diabetes patients, this meal has very high net carbs and may spike blood sugar."
            })
            scores["blood_sugar"] += 2

        elif net_carbs >= RULES["net_carbs_high"]:
            recs.append({
                "type": "warning",
                "message": "For diabetes patients, this meal has high net carbs."
            })
            scores["blood_sugar"] += 1

        else:
            recs.append({
                "type": "positive",
                "message": "For diabetes patients, this meal has controlled net carbs."
            })
            scores["blood_sugar"] -= 1

    # -------- Cholesterol rules --------
    if cholesterol:
        if fat >= RULES["fat_high"]:
            recs.append({
                "type": "warning",
                "message": "For cholesterol patients, this meal has high fat."
            })
            scores["cholesterol"] += 1
        else:
            recs.append({
                "type": "positive",
                "message": "For cholesterol patients, fat levels are acceptable."
            })

    return recs, scores, net_carbs

**Create food-level rule function**

In [13]:
def food_level_recommendations(items):
    recs = []

    for it in items:
        label = it["label"].lower()
        grams = it.get("grams", 0)

        # Rice / high-carb staple
        if "rice" in label:
            recs.append({
                "food": it["label"],
                "type": "warning",
                "message": "Reduce rice portion or switch to brown/red rice to lower glycemic impact."
            })

        # Vegetables
        if any(v in label for v in ["bean", "carrot", "vegetable"]):
            recs.append({
                "food": it["label"],
                "type": "positive",
                "message": "Good vegetable choice; contributes fiber and supports blood sugar and cholesterol control."
            })

        # Protein
        if any(p in label for p in ["shrimp", "fish", "chicken", "egg"]):
            recs.append({
                "food": it["label"],
                "type": "positive",
                "message": "Good lean protein choice; helps balance carbohydrates and improve satiety."
            })

        # Portion-based hint
        if grams > 300:
            recs.append({
                "food": it["label"],
                "type": "recommendation",
                "message": "Large portion detected; consider reducing portion size."
            })

    return recs


**Run Phase 5 rules and print output**

In [14]:
recommendations, scores, net_carbs = apply_rules_for_bloodsugar_cholesterol(
    kcal, carbs, fiber, sugar, protein, fat, diabetes=True,cholesterol=False
)

print("=== Phase 5 Recommendations ===")
for r in recommendations:
    print(r["type"].upper(), ":", r["message"])

print("\nScores:", scores)


=== Phase 5 Recommendations ===
INFO : Meal summary: 441 kcal | Net carbs: 63.7 g | Protein: 29.3 g | Fiber: 15.1 g | Fat: 2.5 g

Scores: {'blood_sugar': 1, 'cholesterol': 0}


**Generate food-level recommendations**

In [15]:
food_recs = food_level_recommendations(items)

print("\n=== Food-level Recommendations ===")
for r in food_recs:
    print(f"{r['food']} → {r['type'].upper()} : {r['message']}")



=== Food-level Recommendations ===
carrot → POSITIVE : Good vegetable choice; contributes fiber and supports blood sugar and cholesterol control.
rice → WARNING : Reduce rice portion or switch to brown/red rice to lower glycemic impact.
French beans → POSITIVE : Good vegetable choice; contributes fiber and supports blood sugar and cholesterol control.
shrimp → POSITIVE : Good lean protein choice; helps balance carbohydrates and improve satiety.


**Meal Rating**

In [16]:
def meal_rating(scores):
    risk = scores["blood_sugar"] + scores["cholesterol"]

    if risk >= 2:
        return "RED (High risk)"
    elif risk >= 1:
        return "AMBER (Moderate risk)"
    else:
        return "GREEN (Low risk)"

rating = meal_rating(scores)
print("\nMeal Rating:", rating)



Meal Rating: AMBER (Moderate risk)


**Save phase5_recommendations.json**

In [17]:
phase5 = {
    "input": "phase4_nutrition.json",

    "meal_rating": rating,

    "scores": scores,
    
    "totals_used": {
        "kcal": kcal,
        "carbs_g": carbs,
        "fiber_g": fiber,
        "net_carbs_g": net_carbs,
        "sugar_g": sugar,
        "protein_g": protein,
        "fat_g": fat,
    },
    "rules": RULES,
    "food_level_recommendations": food_recs,  
    "recommendations": recommendations,
    "notes": [
        "Food-level recommendations are derived from detected food items and portion estimates.",
        "Rules are heuristic and intended for dietary guidance, not medical diagnosis."
    ]
}


OUT_PATH = "/kaggle/working/phase5_recommendations.json"
with open(OUT_PATH, "w") as f:
    json.dump(phase5, f, indent=2)

print("✅ Saved:", OUT_PATH)


✅ Saved: /kaggle/working/phase5_recommendations.json


In [19]:
print("=== PHASE 5 TESTING ===")

# -----------------------------
# Test 1: Function output structure
# -----------------------------
recs, scores_test, net_test = apply_rules_for_bloodsugar_cholesterol(
    kcal=500, carbs=70, fiber=10, sugar=20, protein=25, fat=15,
    diabetes=True, cholesterol=True
)

assert isinstance(recs, list), "Recommendations should be a list"
assert isinstance(scores_test, dict), "Scores should be a dictionary"
assert isinstance(net_test, (int, float)), "Net carbs should be numeric"
print("Test 1 passed: Output structure is correct")


# -----------------------------
# Test 2: Required score keys
# -----------------------------
assert "blood_sugar" in scores_test, "Missing blood_sugar score"
assert "cholesterol" in scores_test, "Missing cholesterol score"
print("Test 2 passed: Score keys are correct")


# -----------------------------
# Test 3: High net carbs rule for diabetes
# -----------------------------
recs, scores_test, net_test = apply_rules_for_bloodsugar_cholesterol(
    kcal=600, carbs=90, fiber=5, sugar=10, protein=25, fat=10,
    diabetes=True
)

msgs = " ".join([r["message"].lower() for r in recs])
assert "very high net carbs" in msgs or "high net carbs" in msgs, "High net carbs warning not triggered"
print("Test 3 passed: Diabetes net carbs warning works")


# -----------------------------
# Test 4: High fat rule for cholesterol
# -----------------------------
recs, scores_test, net_test = apply_rules_for_bloodsugar_cholesterol(
    kcal=500, carbs=40, fiber=5, sugar=10, protein=20, fat=30,
    cholesterol=True
)

msgs = " ".join([r["message"].lower() for r in recs])
assert "high fat" in msgs or "fat levels are acceptable" in msgs, "Cholesterol fat rule not triggered"
print("Test 4 passed: Cholesterol fat rule works")


# -----------------------------
# Test 5: Fiber-related recommendation
# -----------------------------
recs, scores_test, net_test = apply_rules_for_bloodsugar_cholesterol(
    kcal=450, carbs=50, fiber=12, sugar=10, protein=20, fat=10,
    diabetes=True
)

msgs = " ".join([r["message"].lower() for r in recs])
assert "fiber" in msgs, "Fiber-related recommendation missing"
print("Test 5 passed: Fiber rule works")


# -----------------------------
# Test 6: Food-level recommendations
# -----------------------------
sample_items = [
    {"label": "rice", "grams": 200},
    {"label": "carrot", "grams": 80},
    {"label": "shrimp", "grams": 120},
    {"label": "rice", "grams": 350}
]

food_test = food_level_recommendations(sample_items)
assert isinstance(food_test, list), "Food-level recommendations should be a list"

food_msgs = " ".join([r["message"].lower() for r in food_test])
assert "rice" in food_msgs, "Rice rule failed"
assert "fiber" in food_msgs or "vegetable" in food_msgs, "Vegetable rule failed"
assert "protein" in food_msgs or "lean protein" in food_msgs, "Protein rule failed"
assert "large portion" in food_msgs, "Large portion rule failed"
print("Test 6 passed: Food-level rules work")


# -----------------------------
# Test 7: Meal rating logic
# -----------------------------
assert meal_rating({"blood_sugar": 2, "cholesterol": 1}) == "RED (High risk)"
assert meal_rating({"blood_sugar": 1, "cholesterol": 0}) == "AMBER (Moderate risk)"
assert meal_rating({"blood_sugar": -1, "cholesterol": 0}) == "GREEN (Low risk)"
print("Test 7 passed: Meal rating works correctly")


# -----------------------------
# Test 8: Empty items robustness
# -----------------------------
empty_food = food_level_recommendations([])
assert empty_food == [], "Empty items should return an empty list"
print("Test 8 passed: Empty input handled")


# -----------------------------
# Test 9: Zero values robustness
# -----------------------------
recs, scores_test, net_test = apply_rules_for_bloodsugar_cholesterol(
    kcal=0, carbs=0, fiber=0, sugar=0, protein=0, fat=0,
    diabetes=True, cholesterol=True
)

assert net_test == 0, "Net carbs should be 0 for zero inputs"
assert isinstance(recs, list), "Should still return recommendations"
print("Test 9 passed: Zero input handled")


# -----------------------------
# Test 10: JSON export structure
# -----------------------------
assert "meal_rating" in phase5, "meal_rating missing in phase5 JSON"
assert "scores" in phase5, "scores missing in phase5 JSON"
assert "totals_used" in phase5, "totals_used missing in phase5 JSON"
assert "recommendations" in phase5, "recommendations missing in phase5 JSON"
assert "food_level_recommendations" in phase5, "food_level_recommendations missing in phase5 JSON"
print("Test 10 passed: JSON structure is correct")


print("\n✅ All Phase 5 tests completed successfully")

=== PHASE 5 TESTING ===
Test 1 passed: Output structure is correct
Test 2 passed: Score keys are correct
Test 3 passed: Diabetes net carbs warning works
Test 4 passed: Cholesterol fat rule works
Test 5 passed: Fiber rule works
Test 6 passed: Food-level rules work
Test 7 passed: Meal rating works correctly
Test 8 passed: Empty input handled
Test 9 passed: Zero input handled
Test 10 passed: JSON structure is correct

✅ All Phase 5 tests completed successfully
